In [1]:
import os
import re
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("All libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

All libraries imported successfully!
TensorFlow version: 2.20.0


In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Create a folder in your Google Drive to save everything
save_path = '/content/drive/MyDrive/image_caption_generator'
os.makedirs(save_path, exist_ok=True)
print(f"Save folder created at: {save_path}")

Mounted at /content/drive
Save folder created at: /content/drive/MyDrive/image_caption_generator


In [3]:
from google.colab import files

# Upload kaggle.json
files.upload()

# Setup kaggle
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('/content/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 600)

# Download dataset
!pip install kaggle -q
!kaggle datasets download -d adityajn105/flickr8k
!unzip -q flickr8k.zip -d /content/flickr8k

print("Dataset ready!")

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/adityajn105/flickr8k
License(s): CC0-1.0
100% 1.04G/1.04G [00:48<00:00, 23.2MB/s]

Dataset ready!


In [4]:
# Load captions
df = pd.read_csv('/content/flickr8k/captions.txt')
print(f"Total captions: {len(df)}")
print(df.head())

# Clean captions function
def clean_caption(caption):
    # Lowercase
    caption = caption.lower()
    # Remove special characters and numbers
    caption = re.sub(r'[^a-z\s]', '', caption)
    # Remove extra spaces
    caption = re.sub(r'\s+', ' ', caption).strip()
    # Add start and end tokens
    caption = 'startseq ' + caption + ' endseq'
    return caption

# Apply cleaning
df['caption'] = df['caption'].apply(clean_caption)
print("\nCleaned captions sample:")
print(df['caption'].head())

Total captions: 40455
                       image  \
0  1000268201_693b08cb0e.jpg   
1  1000268201_693b08cb0e.jpg   
2  1000268201_693b08cb0e.jpg   
3  1000268201_693b08cb0e.jpg   
4  1000268201_693b08cb0e.jpg   

                                             caption  
0  A child in a pink dress is climbing up a set o...  
1              A girl going into a wooden building .  
2   A little girl climbing into a wooden playhouse .  
3  A little girl climbing the stairs to her playh...  
4  A little girl in a pink dress going into a woo...  

Cleaned captions sample:
0    startseq a child in a pink dress is climbing u...
1    startseq a girl going into a wooden building e...
2    startseq a little girl climbing into a wooden ...
3    startseq a little girl climbing the stairs to ...
4    startseq a little girl in a pink dress going i...
Name: caption, dtype: object


In [5]:
# Get all captions as a list
all_captions = df['caption'].tolist()

# Create tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_captions)

# Vocabulary size
vocab_size = len(tokenizer.word_index) + 1
print(f"Vocabulary size: {vocab_size}")

# Maximum caption length
max_length = max(len(caption.split()) for caption in all_captions)
print(f"Maximum caption length: {max_length} words")

# Show most common words
import collections
word_counts = collections.Counter()
for caption in all_captions:
    word_counts.update(caption.split())

print(f"\nTop 10 most common words:")
for word, count in word_counts.most_common(10):
    print(f"  {word}: {count}")

Vocabulary size: 8781
Maximum caption length: 37 words

Top 10 most common words:
  a: 62986
  startseq: 40455
  endseq: 40455
  in: 18974
  the: 18418
  on: 10743
  is: 9345
  and: 8851
  dog: 8136
  with: 7765


In [6]:
# Load pretrained InceptionV3 model (without top classification layer)
print("Loading InceptionV3 model...")
model = InceptionV3(weights='imagenet')
# Remove the last classification layer
feature_extractor = tf.keras.Model(
    inputs=model.input,
    outputs=model.layers[-2].output
)
print("Model loaded!")
print(f"Feature vector size: {feature_extractor.output_shape}")

Loading InceptionV3 model...
96112376/96112376 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
Model loaded!
Feature vector size: (None, 2048)


In [8]:
# Function to extract features from one image
def extract_features(image_path):
    img = load_img(image_path, target_size=(299, 299))
    img = img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)
    features = feature_extractor.predict(img, verbose=0)
    return features

print("extract_features function defined!")

extract_features function defined!


In [9]:
image_folder = '/content/flickr8k/Images'
image_files = os.listdir(image_folder)

print(f"Extracting features for {len(image_files)} images...")
print("This will take about 10-15 minutes, please wait...")

features = {}
for image_name in tqdm(image_files):
    image_path = os.path.join(image_folder, image_name)
    try:
        feature = extract_features(image_path)
        features[image_name] = feature
    except Exception as e:
        print(f"Error processing {image_name}: {e}")

print(f"\nFeatures extracted for {len(features)} images!")

Extracting features for 8091 images...
This will take about 10-15 minutes, please wait...


100%|██████████| 8091/8091 [14:29<00:00,  9.31it/s]


Features extracted for 8091 images!


In [10]:
# Save features
features_path = os.path.join(save_path, 'features.pkl')
with open(features_path, 'wb') as f:
    pickle.dump(features, f)
print(f"Features saved to Google Drive!")

# Save tokenizer
tokenizer_path = os.path.join(save_path, 'tokenizer.pkl')
with open(tokenizer_path, 'wb') as f:
    pickle.dump(tokenizer, f)
print(f"Tokenizer saved to Google Drive!")

# Save cleaned captions
captions_path = os.path.join(save_path, 'cleaned_captions.csv')
df.to_csv(captions_path, index=False)
print(f"Cleaned captions saved to Google Drive!")

print(f"\nSummary:")
print(f"  Total images processed: {len(features)}")
print(f"  Vocabulary size: {vocab_size}")
print(f"  Max caption length: {max_length}")

Features saved to Google Drive!
Tokenizer saved to Google Drive!
Cleaned captions saved to Google Drive!

Summary:
  Total images processed: 8091
  Vocabulary size: 8781
  Max caption length: 37


In [11]:
files_saved = os.listdir(save_path)
print("Files saved in Google Drive:")
for f in files_saved:
    size = os.path.getsize(os.path.join(save_path, f))
    print(f"  {f} — {size/1024/1024:.1f} MB")

Files saved in Google Drive:
  features.pkl — 63.7 MB
  tokenizer.pkl — 0.3 MB
  cleaned_captions.csv — 3.7 MB
